In [19]:
import pandas as pd
import glob
import os

# Path to the parent folder containing subfolders of csv files
base_path = "software/debug/sessions_files/workspace"
base_path2 = "software/debug/sessions_files/track_trajectory"

# Get all csv files from subfolders
csv_1 = glob.glob(os.path.join(base_path, "**", "*.csv"), recursive=True)
csv_2 = glob.glob(os.path.join(base_path2, "**", "*.csv"), recursive=True)

csvs = [csv_1, csv_2]
# print(csvs)

# Dictionary to hold DataFrames with custom names
dataframes = {}

# print(csv_files)

for csv in csvs:
    for file in csv:
        folder = os.path.basename(os.path.dirname(file))   # parent folder name
        filename = os.path.splitext(os.path.basename(file))[0]  # csv name without extension
        df_name = f"{folder}_{filename}"
    
        dataframes[df_name] = pd.read_csv(file)

# Example: access one DataFrame
keys = list(dataframes.keys())
print(keys)   # shows all dataframe names
print(dataframes[keys[0]].head())

['workspace_1', 'workspace_10', 'workspace_11', 'workspace_12', 'workspace_13', 'workspace_14', 'workspace_15', 'workspace_16', 'workspace_17', 'workspace_18', 'workspace_19', 'workspace_2', 'workspace_20', 'workspace_21', 'workspace_22', 'workspace_3', 'workspace_4', 'workspace_5', 'workspace_6', 'workspace_7', 'workspace_8', 'workspace_9', 'track_trajectory_1', 'track_trajectory_10', 'track_trajectory_11', 'track_trajectory_12', 'track_trajectory_13', 'track_trajectory_14', 'track_trajectory_15', 'track_trajectory_16', 'track_trajectory_17', 'track_trajectory_18', 'track_trajectory_19', 'track_trajectory_2', 'track_trajectory_20', 'track_trajectory_3', 'track_trajectory_4', 'track_trajectory_5', 'track_trajectory_6', 'track_trajectory_7', 'track_trajectory_8', 'track_trajectory_9']
   THRESHOLD   SCORE   UPPER ANGLE   UPPER THRESHOLD   UPPER_LOAD_CELL  \
0          0       0             4                 0             32257   
1          0       0             4                 0     

In [20]:
import numpy as np
import pandas as pd

def process_dataframe(df, l2=26, l3=46):
    """
    Process a dataframe:
    1. Drop unwanted columns.
    2. Compute x and y from LOWER ANGLE and UPPER ANGLE.
    3. Return cleaned dataframe.
    """

    # Identify columns dynamically (based on your passive_6 example)
    cols_to_drop = [df.columns[0], df.columns[3], df.columns[6],
                    df.columns[8], df.columns[9], df.columns[10], df.columns[11]]
    # print("Columns to Drop: ", cols_to_drop) # uncomment for debugging
    
    df = df.drop(columns=cols_to_drop, errors="ignore")

    # Convert angles to radians
    theta1 = np.radians(df[' LOWER ANGLE'])
    theta2 = np.radians(df[' UPPER ANGLE'])

    # Compute x and y
    df['x'] = l2 * np.cos(theta2) + l3 * np.cos(theta1)
    df['y'] = l2 * np.sin(theta2) + l3 * np.sin(theta1)

    return df


In [21]:
# Apply process_dataframe to all DataFrames in the dataframes dictionary
processed_dataframes = {}

for name, df in dataframes.items():
    processed_dataframes[name] = process_dataframe(df, l2=26, l3=46)

# Example: print keys and first few rows of one processed DataFrame
print(list(processed_dataframes.keys()))

['workspace_1', 'workspace_10', 'workspace_11', 'workspace_12', 'workspace_13', 'workspace_14', 'workspace_15', 'workspace_16', 'workspace_17', 'workspace_18', 'workspace_19', 'workspace_2', 'workspace_20', 'workspace_21', 'workspace_22', 'workspace_3', 'workspace_4', 'workspace_5', 'workspace_6', 'workspace_7', 'workspace_8', 'workspace_9', 'track_trajectory_1', 'track_trajectory_10', 'track_trajectory_11', 'track_trajectory_12', 'track_trajectory_13', 'track_trajectory_14', 'track_trajectory_15', 'track_trajectory_16', 'track_trajectory_17', 'track_trajectory_18', 'track_trajectory_19', 'track_trajectory_2', 'track_trajectory_20', 'track_trajectory_3', 'track_trajectory_4', 'track_trajectory_5', 'track_trajectory_6', 'track_trajectory_7', 'track_trajectory_8', 'track_trajectory_9']


In [22]:
def final_processing(df):
    """
    Process a dataframe:
    1. Drop more unwanted columns.
    2. Return cleaned dataframe.
    """

    # Identify columns dynamically (based on your passive_6 example)
    cols_to_drop = [df.columns[0], df.columns[1], df.columns[2],
                    df.columns[3], df.columns[4]]
    # print("Columns to Drop: ", cols_to_drop) # uncomment for debugging
    
    df = df.drop(columns=cols_to_drop, errors="ignore")

    return df

In [23]:
final_processing(processed_dataframes['workspace_1'].copy())

,x,y
0,-1.746826,38.550902
1,-1.746826,38.550902
2,-1.746826,38.550902
3,-1.746826,38.550902
4,-1.746826,38.550902
...,...,...
234,10.267073,43.225861
235,10.267073,43.225861
236,10.267073,43.225861
237,10.267073,43.225861


In [24]:
# Apply on all df
final_dataframes = {}
for name, df in processed_dataframes.items():
    final_dataframes[name] = final_processing(df)

In [25]:
final_dataframes['workspace_1'].head()

,x,y
0,-1.746826,38.550902
1,-1.746826,38.550902
2,-1.746826,38.550902
3,-1.746826,38.550902
4,-1.746826,38.550902


In [26]:
# Save all processed DataFrames as CSV files
output_workspace = "workspace_coordinates"
output_track = "track_trajectory_coordinates"
outputs = [output_workspace, output_track]

for output in outputs:
    os.makedirs(output, exist_ok=True)

for name, df in final_dataframes.items():
    if "workspace" in name:
        csv_path = os.path.join(output_workspace, f"{name}.csv")
        df.to_csv(csv_path, index=False)
    elif "track" in name:
        csv_path = os.path.join(output_track, f"{name}.csv")
        df.to_csv(csv_path, index=False)